# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A1704/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Research paper findings and methodology questions

**Research paper:** *Content Refresh & Opportunity Scoring: Prioritizing Enterprise Editorial Resources via Supervised Decay Modeling*

**Author:** R. Pranitha, FlyRank Applied Machine Learning Research, September 2026.

**Source:** https://pranitha-r.github.io/flyrank-8.0/

### Finding 1: Client-grouped validation provides a more realistic test of generalization.

The paper uses a client-holdout partition, training on 25 client domains and evaluating on seven completely held-out domains. This design is intended to prevent the same client domains from appearing in both training and test sets. The selected Random Forest model reports a Precision@50 of 74.0% and ROC-AUC of 0.750 on the holdout set.

**Methodology question:** How were the seven test client domains selected? Were differences in client characteristics and label distributions considered when creating the split, and how sensitive were the results to the particular domains held out?

### Finding 2: Predicting historical content decline does not establish that refreshing content causes improvement.

The paper explicitly states that its model measures associations in historical search-performance data. It cannot establish that refreshing a page causes traffic recovery without randomized A/B testing.

**Methodology question:** Was the model evaluated against actual outcomes after content refreshes, or only against historical declining-trend labels? What additional experimental design would be required to measure whether model-selected refreshes produce an improvement in traffic or engagement?

### Relevance to my Week 5 model

My model also uses the 30,000-row, 32-client dataset and a client-grouped validation split. This paper provides a useful methodological reference for evaluating performance on unseen clients and distinguishing historical prediction from causal impact.

The paper's reported metrics belong to its own experiment and should not be presented as results from my Week 5 model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compared Logistic Regression using a random row-level split and an 80/20 client-grouped split, keeping the feature preparation and Precision@50 metric consistent. The random split achieved Precision@50 of 0.880, average precision of 0.727, and ROC AUC of 0.711. However, 31 of the 32 clients appeared in both training and test sets. Under the client-grouped split, Precision@50 was 0.720, average precision was 0.605, and ROC AUC was 0.615, with zero client overlap. The lower grouped-split scores suggest that the random split may provide an optimistic estimate when evaluating generalization to unseen clients. The grouped result is more relevant for that purpose. The difference should be interpreted cautiously because the two validation methods also use different test samples. These results measure prediction of historical declining-trend labels, not the causal impact of refreshing content.

In [2]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("\nFiles in /content:")

for item in Path("/content").iterdir():
    print(item)

Current directory: /content

Files in /content:
/content/.config
/content/sample_data


In [3]:
!git clone https://github.com/A1704/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 167 (delta 71), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 1.92 MiB | 9.60 MiB/s, done.
Resolving deltas: 100% (71/71), done.


In [4]:
from pathlib import Path

DATA_PATH = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Dataset size (MB):", round(
        DATA_PATH.stat().st_size / (1024 * 1024), 2
    ))
else:
    print("Dataset not found. We need to inspect the repository folders.")

Dataset exists: True
Dataset size (MB): 6.42


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# --------------------------------------------------
# 1. Load and prepare the same dataset used in Week 5
# --------------------------------------------------

DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Week 5 dataset not found. Check the Colab Files panel.")

df = pd.read_csv(DATA_PATH)

df["trend_direction"] = df["trend_direction"].fillna("unknown").astype(str)
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

df = df[
    (pd.to_numeric(df["impressions_90d"], errors="coerce") > 0)
    & (pd.to_numeric(df["content_age_days"], errors="coerce") >= 90)
].drop_duplicates("content_id").reset_index(drop=True)

# Exclude target-related columns and identifiers.
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

log_source = {
    "log_impressions_90d": "impressions_90d",
    "log_clicks_90d": "clicks_90d",
    "log_sessions_90d": "sessions_90d",
    "log_ai_sessions_90d": "ai_sessions_90d",
}

for new_col, source_col in log_source.items():
    if source_col in df.columns:
        df[new_col] = np.log1p(
            pd.to_numeric(df[source_col], errors="coerce").clip(lower=0)
        )
        numeric_features.append(new_col)

categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

missing_flag_features = []
for c in numeric_features:
    flag = f"has_value_{c}"
    df[flag] = df[c].notna().astype(int)
    missing_flag_features.append(flag)

feature_columns = numeric_features + categorical_features + missing_flag_features

X = df[feature_columns].copy()
y = df["is_declining_label"].astype(int)
groups = df["client_id"].fillna("unknown").astype(str)

# --------------------------------------------------
# 2. Define one consistent model and evaluation metric
# --------------------------------------------------

def make_model():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer,
         numeric_features + missing_flag_features),
        ("categorical", categorical_transformer, categorical_features)
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

def precision_at_50(y_true, scores):
    y_array = np.asarray(y_true)
    scores_array = np.asarray(scores)
    top_indices = np.argsort(-scores_array, kind="stable")[:min(50, len(y_array))]
    return float(y_array[top_indices].mean())

# --------------------------------------------------
# 3. BEFORE: Random row-level 80/20 split
# --------------------------------------------------

row_train_idx, row_test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

row_model = make_model()
row_model.fit(X.iloc[row_train_idx], y.iloc[row_train_idx])

row_scores = row_model.predict_proba(X.iloc[row_test_idx])[:, 1]

row_overlap = len(
    set(groups.iloc[row_train_idx]) & set(groups.iloc[row_test_idx])
)

# --------------------------------------------------
# 4. AFTER: Client-grouped 80/20 split
# --------------------------------------------------

group_split_found = False

for seed in range(42, 142):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed
    )

    group_train_idx, group_test_idx = next(
        splitter.split(X, y, groups)
    )

    if (
        y.iloc[group_train_idx].nunique() == 2
        and y.iloc[group_test_idx].nunique() == 2
    ):
        group_split_found = True
        break

if not group_split_found:
    raise ValueError("Could not find a valid client-grouped split.")

group_overlap = len(
    set(groups.iloc[group_train_idx]) &
    set(groups.iloc[group_test_idx])
)

assert group_overlap == 0

group_model = make_model()
group_model.fit(X.iloc[group_train_idx], y.iloc[group_train_idx])

group_scores = group_model.predict_proba(X.iloc[group_test_idx])[:, 1]

# --------------------------------------------------
# 5. Compare measured results
# --------------------------------------------------

comparison = pd.DataFrame([
    {
        "Validation design": "Random row split (before)",
        "Training rows": len(row_train_idx),
        "Test rows": len(row_test_idx),
        "Client overlap": row_overlap,
        "Precision@50": precision_at_50(
            y.iloc[row_test_idx], row_scores
        ),
        "Average precision": average_precision_score(
            y.iloc[row_test_idx], row_scores
        ),
        "ROC AUC": roc_auc_score(
            y.iloc[row_test_idx], row_scores
        )
    },
    {
        "Validation design": "Client-grouped split (after)",
        "Training rows": len(group_train_idx),
        "Test rows": len(group_test_idx),
        "Client overlap": group_overlap,
        "Precision@50": precision_at_50(
            y.iloc[group_test_idx], group_scores
        ),
        "Average precision": average_precision_score(
            y.iloc[group_test_idx], group_scores
        ),
        "ROC AUC": roc_auc_score(
            y.iloc[group_test_idx], group_scores
        )
    }
])

print("Dataset rows:", len(df))
print("Unique clients:", groups.nunique())
print("\nValidation comparison:")
display(comparison.round(3))

print("\nInterpretation:")
print("The random row split has", row_overlap,
      "clients appearing in both training and test sets.")
print("The client-grouped split has", group_overlap,
      "clients appearing in both training and test sets.")
print("Compare the measured metrics rather than assuming either split "
      "must produce a higher score.")



Dataset rows: 30000
Unique clients: 32

Validation comparison:


,Validation design,Training rows,Test rows,Client overlap,Precision@50,Average precision,ROC AUC
0,Random row split (before),24000,6000,31,0.88,0.727,0.711
1,Client-grouped split (after),23837,6163,0,0.72,0.605,0.615



Interpretation:
The random row split has 31 clients appearing in both training and test sets.
The client-grouped split has 0 clients appearing in both training and test sets.
Compare the measured metrics rather than assuming either split must produce a higher score.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the 44-feature Week 5 model input for direct and potential indirect target leakage. The target, `is_declining_label`, is derived from `trend_direction`, so I verified that `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from the model features. Identifiers such as `client_id` and `content_id` are also excluded from prediction.

The preprocessing pipeline keeps imputation, scaling, and categorical encoding inside the pipeline, so these transformations are fitted as part of model training. The client-grouped split also has zero client overlap between training and test sets.

I inspected five potentially derived categorical features: `freshness_tier`, `impression_tier`, `position_tier`, `age_tier`, and `word_count_tier`. Their observed category ranges align with the corresponding numeric fields (`days_since_last_update`, `impressions_90d`, `avg_position`, `content_age_days`, and `word_count`). This is consistent with these features being bucketed versions of the numeric fields.

However, the notebook and diagnostic output do not establish the original feature-generation process or confirm that every feature was available before the prediction point. Therefore, I cannot claim that all indirect leakage has been ruled out. The tier features require provenance and timing confirmation before the feature set can be considered fully validated.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 3 — Leakage audit of the final Week 5 feature set

import pandas as pd

# 1. Check whether direct target and identifier columns are features.
target_related_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

identifier_columns = [
    "client_id",
    "content_id"
]

print("TOTAL MODEL FEATURES:", len(feature_columns))

print("\n1. DIRECT TARGET-LEAKAGE CHECK")
for column in target_related_columns:
    print(
        f"{column}:",
        "PRESENT — INVESTIGATE" if column in feature_columns
        else "Excluded from model features"
    )

print("\n2. IDENTIFIER CHECK")
for column in identifier_columns:
    print(
        f"{column}:",
        "PRESENT — INVESTIGATE" if column in feature_columns
        else "Excluded from model features"
    )

# 2. Display the complete feature list.
print("\n3. FINAL FEATURE LIST")
display(pd.DataFrame({
    "Feature": feature_columns,
    "Feature type": [
        "Numeric" if c in numeric_features + missing_flag_features
        else "Categorical"
        for c in feature_columns
    ]
}))

# 3. Flag features that may need source/timing verification.
potentially_derived = [
    "freshness_tier",
    "impression_tier",
    "position_tier",
    "age_tier",
    "word_count_tier"
]

print("\n4. POTENTIALLY DERIVED FEATURES")
for column in potentially_derived:
    if column in feature_columns:
        print(f"{column}: Present — verify how and when it was created.")
    else:
        print(f"{column}: Not used.")

# 4. Confirm the train/test client separation.
train_clients = set(groups.iloc[group_train_idx])
test_clients = set(groups.iloc[group_test_idx])

print("\n5. CLIENT GROUP SEPARATION")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

# 5. Confirm preprocessing is inside the model pipeline.
print("\n6. PREPROCESSING CHECK")
print("Preprocessor step:", list(group_model.named_steps.keys()))
print(
    "Imputer, scaler and encoder are inside the pipeline:",
    "preprocessor" in group_model.named_steps
)

# 6. Final automated checks.
assert not any(c in feature_columns for c in target_related_columns)
assert not any(c in feature_columns for c in identifier_columns)
assert len(train_clients & test_clients) == 0
assert "preprocessor" in group_model.named_steps

print("\nAUDIT CHECKS PASSED.")
print("Potentially derived features still require provenance review.")


# 7. Inspect how tier features relate to their underlying numeric fields.
tier_sources = {
    "freshness_tier": "days_since_last_update",
    "impression_tier": "impressions_90d",
    "position_tier": "avg_position",
    "age_tier": "content_age_days",
    "word_count_tier": "word_count"
}

for tier, source in tier_sources.items():
    print(f"\n{tier} compared with {source}")

    if tier in df.columns and source in df.columns:
        summary = (
            df.groupby(tier, dropna=False)[source]
            .agg(["count", "min", "max"])
            .reset_index()
        )
        display(summary)
    else:
        print("Tier or source column not available.")




TOTAL MODEL FEATURES: 44

1. DIRECT TARGET-LEAKAGE CHECK
trend_direction: Excluded from model features
trend_pct: Excluded from model features
is_declining_label: Excluded from model features

2. IDENTIFIER CHECK
client_id: Excluded from model features
content_id: Excluded from model features

3. FINAL FEATURE LIST


,Feature,Feature type
0,search_volume,Numeric
1,competition,Numeric
2,cpc,Numeric
3,word_count,Numeric
4,char_count,Numeric
5,days_with_impressions,Numeric
6,days_with_sessions,Numeric
7,content_age_days,Numeric
8,days_since_last_update,Numeric
9,ctr,Numeric



4. POTENTIALLY DERIVED FEATURES
freshness_tier: Present — verify how and when it was created.
impression_tier: Present — verify how and when it was created.
position_tier: Present — verify how and when it was created.
age_tier: Present — verify how and when it was created.
word_count_tier: Present — verify how and when it was created.

5. CLIENT GROUP SEPARATION
Training clients: 25
Test clients: 7
Client overlap: 0

6. PREPROCESSING CHECK
Preprocessor step: ['preprocessor', 'classifier']
Imputer, scaler and encoder are inside the pipeline: True

AUDIT CHECKS PASSED.
Potentially derived features still require provenance review.

freshness_tier compared with days_since_last_update


,freshness_tier,count,min,max
0,0-30,20480,1,28
1,181+,174,183,373
2,31-90,175,34,89
3,91-180,9171,92,151



impression_tier compared with impressions_90d


,impression_tier,count,min,max
0,excellent,1078,30002,517715
1,good,7205,3001,29981
2,low,11248,1,299
3,moderate,10469,300,2999


... (2 outputs left out)


age_tier compared with content_age_days


,age_tier,count,min,max
0,181-365,11368,181,362
1,31-90,492,90,90
2,365+,6360,372,564
3,91-180,11780,91,180



word_count_tier compared with word_count


,word_count_tier,count,min,max
0,1000-2000,3780,1001.0,1998.0
1,2000-3500,11263,2000.0,3499.0
2,3500+,6285,3501.0,9546.0
3,<1000,973,8.0,999.0
4,NaN,0,NaN,NaN


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim to avoid

The model accurately identifies which content should be refreshed and can predict which refreshes will improve performance.

### Revised evidence-based claim

The Logistic Regression model showed directional decision-support potential for identifying content associated with historical declining trends. Under an 80/20 client-grouped validation split with zero client overlap, it achieved Precision@50 of 0.720, average precision of 0.605, and ROC AUC of 0.615.

The random row-level split produced a higher Precision@50 of 0.880, but 31 of 32 clients appeared in both training and test sets. This suggests that a random row split may give an optimistic estimate when the intended use is generalization to unseen clients.

These results measure the model's ability to identify historical declining-trend labels in this dataset. They do not demonstrate that refreshing selected content will cause traffic or engagement to improve. The model should therefore be treated as a decision-support aid, not an automated refresh recommendation or a guarantee of improvement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

### Validation audit self-check

- [x] Compared a random row-level split with a client-grouped split.
- [x] Confirmed zero client overlap in the grouped split.
- [x] Checked that direct target columns and identifiers are excluded from model features.
- [x] Confirmed preprocessing is fitted inside the model pipeline.
- [x] Inspected potentially derived tier features against their numeric source fields.
- [x] Documented that feature-generation timing and provenance still require confirmation.
- [x] Rewrote the model claim to reflect measured results and avoid causal claims.
- [X] Added two findings and methodology questions from the assigned research paper.
- [ ] Restarted the runtime and ran the notebook from top to bottom.
- [ ] Saved the executed notebook and committed it to the GitHub repository.

**Remaining limitation:** The research-paper findings and feature-generation provenance have not yet been independently verified. The audit therefore documents these gaps rather than claiming complete validation.